In [ ]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(comments_df["clean_comment"])
sequences = tokenizer.texts_to_sequences(comments_df["clean_comment"])
word_index = tokenizer.word_index

MAX_LEN = min(100, max(len(seq) for seq in sequences))
padded_sequences = pad_sequences(sequences, maxlen=MAX_LEN, padding='post')

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test = train_test_split(padded_sequences, test_size=0.2, random_state=42)

X_train = np.array(X_train)
X_test = np.array(X_test)

Y_train = X_train[:, 1:]  # Shift left
X_train = X_train[:, :-1]  # Shift right

Y_test = X_test[:, 1:]
X_test = X_test[:, :-1]

vocab_size = len(word_index) + 1  # Tổng số từ trong từ điển
embedding_dim = 128
sequence_len = MAX_LEN

model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=sequence_len),
    Bidirectional(LSTM(64, return_sequences=True, dropout=0.3)),
    Dense(vocab_size, activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)


history = model.fit(
    X_train, Y_train,
    epochs=20,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stop]
)

loss, acc = model.evaluate(X_test, Y_test)
print(f"\nTest Accuracy: {acc}")
